# 04 — T2 Tuning: Optuna cho M2a XGBoost + M2b LightGBM

exp_003 — [RESULTS chi tiết](../experiments/exp_003_tuning_m2/). Tune CẢ HAI model (không chỉ 1, xem docstring `run.py` — ở exp_002 hai model gần như hoà nhau).

**Đã sanity-check bằng 5 trial thật**: XGBoost ~26s/trial, LightGBM ~7.5s/trial. Với 100 trial mỗi model, tổng thời gian ước tính **~1 giờ** (XGBoost ~43 phút + LightGBM ~12-13 phút). Không cần GPU bắt buộc (CPU đủ nhanh ở quy mô dữ liệu này) — máy bạn có GPU nên chạy thoải mái, không lo treo máy.

Đổi `N_TRIALS` xuống thấp (vd 10) để chạy thử nhanh trước khi chạy full 100.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "app").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("project root:", _root)

In [ ]:
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(_root / 'experiments' / 'exp_003_tuning_m2'))
from run import (
    HORIZONS,
    INNER_CUTOFF,
    INNER_EMBARGO_MONTHS,
    INNER_N_ORIGINS,
    OUTER_EMBARGO_MONTHS,
    OUTER_N_ORIGINS,
    REPORTING_DELAY_MONTHS,
    build_horizon_pairs,
    evaluate_params,
    load_real_panel_with_features,
    tune,
)

from app.forecast.models import fit_predict_m2_lightgbm, fit_predict_m2_xgboost
from app.forecast.splits import make_splits

N_TRIALS = 100

## Chuẩn bị dữ liệu (inner tuning + outer đánh giá, tách biệt hoàn toàn)

In [ ]:
feat_panel = load_real_panel_with_features()

inner_panel = feat_panel[feat_panel['month'] <= INNER_CUTOFF]
inner_splits = make_splits(inner_panel, n_origins=INNER_N_ORIGINS, horizons=HORIZONS,
                            embargo_months=INNER_EMBARGO_MONTHS, reporting_delay_months=REPORTING_DELAY_MONTHS)
inner_cache = {h: build_horizon_pairs(inner_panel, h) for h in HORIZONS}

outer_splits = make_splits(feat_panel, n_origins=OUTER_N_ORIGINS, horizons=HORIZONS,
                            embargo_months=OUTER_EMBARGO_MONTHS, reporting_delay_months=REPORTING_DELAY_MONTHS)
outer_cache = {h: build_horizon_pairs(feat_panel, h) for h in HORIZONS}

print(f'Inner: {len(inner_splits)} origin (tune). Outer: {len(outer_splits)} origin (danh gia cuoi).')

## Tune M2a XGBoost

Cell này chạy lâu nhất (~43 phút với 100 trial) — cứ để chạy, không cần theo dõi liên tục.

In [ ]:
t0 = time.time()
study_xgb = tune(fit_predict_m2_xgboost, inner_panel, inner_splits, inner_cache, N_TRIALS)
print(f'Xong sau {time.time()-t0:.0f}s')
print('Best inner MASE:', study_xgb.best_value)
print('Best params:', study_xgb.best_params)

## Tune M2b LightGBM

In [ ]:
t0 = time.time()
study_lgb = tune(fit_predict_m2_lightgbm, inner_panel, inner_splits, inner_cache, N_TRIALS)
print(f'Xong sau {time.time()-t0:.0f}s')
print('Best inner MASE:', study_lgb.best_value)
print('Best params:', study_lgb.best_params)

## Đường hội tụ (docs/02 §6.2 yêu cầu)

Nếu đường phẳng từ khoảng trial 20-30 trở đi thì tăng ngân sách trial thêm là lãng phí — nhìn biểu đồ để biết 100 trial có thực sự cần thiết không.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, study, name in [(axes[0], study_xgb, 'XGBoost'), (axes[1], study_lgb, 'LightGBM')]:
    values = [t.value for t in study.trials]
    best_so_far = np.minimum.accumulate(values)
    ax.plot(values, 'o', alpha=0.3, label='moi trial')
    ax.plot(best_so_far, '-', linewidth=2, label='tot nhat tinh den luc do')
    ax.set_title(f'{name}: hoi tu qua {len(values)} trial')
    ax.set_xlabel('trial'); ax.set_ylabel('inner MASE (avg 5 origin x horizon)')
    ax.legend()
plt.tight_layout()
plt.show()

## Đánh giá cuối: T1 mặc định vs T2 đã tune, trên 8 outer origin (giống hệt exp_002)

In [ ]:
rows = []
for name, fit_fn, study in [
    ('M2a_xgboost', fit_predict_m2_xgboost, study_xgb),
    ('M2b_lightgbm', fit_predict_m2_lightgbm, study_lgb),
]:
    t1 = evaluate_params(fit_fn, None, feat_panel, outer_splits, outer_cache)
    t2 = evaluate_params(fit_fn, study.best_params, feat_panel, outer_splits, outer_cache)
    rows.append({'model': name, 'T1_default_MASE': round(t1, 4), 'T2_tuned_MASE': round(t2, 4),
                 'cai_thien_%': round(100*(t1-t2)/t1, 1)})

import pandas as pd

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## Lưu kết quả

In [ ]:
import json

out = {
    'xgboost': {'n_trials': N_TRIALS, 'best_inner_mase': study_xgb.best_value,
                'best_params': study_xgb.best_params,
                'convergence': [t.value for t in study_xgb.trials]},
    'lightgbm': {'n_trials': N_TRIALS, 'best_inner_mase': study_lgb.best_value,
                 'best_params': study_lgb.best_params,
                 'convergence': [t.value for t in study_lgb.trials]},
    'outer_comparison': rows,
}
out_path = _root / 'experiments' / 'exp_003_tuning_m2' / f'results_trials{N_TRIALS}.json'
out_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Da luu vao {out_path}')

**Xong.** Báo lại cho tôi (Claude) kết quả — tôi sẽ đọc `results_trials100.json`, viết `RESULTS.md` cho exp_003, và cập nhật PROGRESS_LOG.md.